# Enhanced Heart Disease Prediction — Review-Ready Notebook

This notebook is the guided review version of the project. It uses the same leakage-safe training script and saved pipeline as the Streamlit application.

**Important:** Educational prototype only; not a medical diagnostic tool. The supplied CSV is used as-is with exact duplicate rows removed during training.

## 1. Dataset audit

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "heart.csv"
df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
display(df.head())
display(df["target"].value_counts().rename("count").to_frame())

## 2. Train, compare and validate models
This executes the reproducible pipeline. Preprocessing is fit inside each training fold to reduce leakage.

In [ ]:
%run train_model.py

## 3. Compare model metrics

In [ ]:
comparison = pd.read_csv(ROOT / "models" / "model_comparison.csv")
display(comparison.round(3))

## 4. Evaluation plots

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT / "models" / "confusion_matrix.png")))
display(Image(filename=str(ROOT / "models" / "roc_curve.png")))

## 5. Smoke-test a prediction with the saved pipeline

In [ ]:
import joblib
model = joblib.load(ROOT / "models" / "heart_disease_pipeline.joblib")
sample = df.drop_duplicates().drop(columns="target").iloc[[0]]
prediction = int(model.predict(sample)[0])
probability_target_1 = float(model.predict_proba(sample)[0, 1])
print("Predicted dataset class:", prediction)
print(f"Estimated model probability for target=1: {probability_target_1:.1%}")
print("This probability is not a clinical risk estimate or diagnosis.")

## 6. Limitations to explain in the review

- The provided 1,025-row CSV contains 723 exact duplicate rows; only 302 unique rows remain.
- The held-out test set has 61 rows, so metrics are uncertain and split-dependent.
- Dataset provenance, representativeness, external validation, calibration and subgroup fairness have not been established.
- The model must not be used to diagnose or rule out heart disease.
